# M5 Store-Department Weather Data Quality Assessment

Notebook này kiểm tra chất lượng dữ liệu cho file M5 đã join weather:

```text
data/m5-forecasting-accuracy/processed/m5_store_dept_daily_with_weather.csv
```

Mục tiêu của notebook là kiểm tra trước EDA/modeling:

1. Schema.
2. Missing values.
3. Duplicate và key uniqueness.
4. Range và giá trị bất thường.
5. Data quality assessment.
6. Bias, limitation và variation assessment.

Notebook này không train model và không tạo feature lag/rolling. Mục tiêu là xác nhận dữ liệu đủ đáng tin để bước sau mới làm EDA và tạo model-ready dataset.

## 1. Setup và đọc dữ liệu

Đầu tiên cần đọc file processed đã join weather và xác nhận file tồn tại, shape, khoảng thời gian, grain dự kiến.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_PATH = Path("data/m5-forecasting-accuracy/processed/m5_store_dept_daily_with_weather.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file input: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"], low_memory=False)

grain_cols = ["date", "store_id", "dept_id"]
target_col = "daily_revenue"
weather_feature_cols = [
    "weather_code",
    "temperature_max_c",
    "temperature_min_c",
    "temperature_mean_c",
    "apparent_temperature_mean_c",
    "precipitation_mm",
    "rain_mm",
    "snowfall_cm",
    "wind_speed_max_kmh",
    "wind_gusts_max_kmh",
    "shortwave_radiation_mj_m2",
]
event_cols = ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
leakage_risk_cols = ["daily_units", "active_item_count", "has_sales", "weighted_avg_sell_price"]

overview = pd.DataFrame({
    "metric": [
        "file_path",
        "rows",
        "columns",
        "date_min",
        "date_max",
        "unique_dates",
        "unique_states",
        "unique_stores",
        "unique_categories",
        "unique_departments",
    ],
    "value": [
        str(DATA_PATH),
        df.shape[0],
        df.shape[1],
        df["date"].min(),
        df["date"].max(),
        df["date"].nunique(),
        df["state_id"].nunique(),
        df["store_id"].nunique(),
        df["cat_id"].nunique(),
        df["dept_id"].nunique(),
    ],
})

display(overview)
display(df.head())

print(f"Đã đọc file: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print(f"Grain kỳ vọng: {' + '.join(grain_cols)}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(
    f"Coverage: {df['state_id'].nunique()} bang, "
    f"{df['store_id'].nunique()} stores, "
    f"{df['dept_id'].nunique()} departments"
)

,metric,value
0,file_path,data/m5-forecasting-accuracy/processed/m5_stor...
1,rows,135870
2,columns,56
3,date_min,2011-01-29 00:00:00
4,date_max,2016-05-22 00:00:00
5,unique_dates,1941
6,unique_states,3
7,unique_stores,10
8,unique_categories,3
9,unique_departments,7


,date,d,wm_yr_wk,store_id,state_id,cat_id,dept_id,daily_revenue,daily_units,item_count,active_item_count,weighted_avg_sell_price,has_sales,weekday,wday,month,year,quarter,week_of_year,day_of_month,day_of_year,day_of_week,day_of_week_num,is_weekend,year_month,event_name_1,event_type_1,event_name_2,event_type_2,event_count,snap_CA,snap_TX,snap_WI,snap_active,weather_location_id,location_name,weather_spatial_level,weather_code,temperature_max_c,temperature_min_c,temperature_mean_c,apparent_temperature_mean_c,precipitation_mm,rain_mm,snowfall_cm,wind_speed_max_kmh,wind_gusts_max_kmh,shortwave_radiation_mj_m2,latitude_requested,longitude_requested,latitude_open_meteo,longitude_open_meteo,elevation_m,timezone,utc_offset_seconds,weather_source
0,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_1,681.1800,297,216,70,2.2935,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,NaN,NaN,NaN,NaN,0,0,0,0,0,m5_CA,"Los Angeles, CA",state_representative_city,3,18.1000,6.1000,11.7000,9.7000,0.0000,0.0000,0.0000,10.9000,25.9000,14.8100,34.0522,-118.2437,34.0598,-118.2375,91.0000,America/Los_Angeles,-25200,open_meteo_archive
1,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_2,"2,236.0100",674,398,154,3.3175,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,NaN,NaN,NaN,NaN,0,0,0,0,0,m5_CA,"Los Angeles, CA",state_representative_city,3,18.1000,6.1000,11.7000,9.7000,0.0000,0.0000,0.0000,10.9000,25.9000,14.8100,34.0522,-118.2437,34.0598,-118.2375,91.0000,America/Los_Angeles,-25200,open_meteo_archive
2,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_3,"4,323.4600",2268,823,285,1.9063,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,NaN,NaN,NaN,NaN,0,0,0,0,0,m5_CA,"Los Angeles, CA",state_representative_city,3,18.1000,6.1000,11.7000,9.7000,0.0000,0.0000,0.0000,10.9000,25.9000,14.8100,34.0522,-118.2437,34.0598,-118.2375,91.0000,America/Los_Angeles,-25200,open_meteo_archive
3,2011-01-29,d_1,11101,CA_1,CA,HOBBIES,HOBBIES_1,"1,276.8600",528,416,101,2.4183,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,NaN,NaN,NaN,NaN,0,0,0,0,0,m5_CA,"Los Angeles, CA",state_representative_city,3,18.1000,6.1000,11.7000,9.7000,0.0000,0.0000,0.0000,10.9000,25.9000,14.8100,34.0522,-118.2437,34.0598,-118.2375,91.0000,America/Los_Angeles,-25200,open_meteo_archive
4,2011-01-29,d_1,11101,CA_1,CA,HOBBIES,HOBBIES_2,93.0500,28,149,17,3.3232,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,NaN,NaN,NaN,NaN,0,0,0,0,0,m5_CA,"Los Angeles, CA",state_representative_city,3,18.1000,6.1000,11.7000,9.7000,0.0000,0.0000,0.0000,10.9000,25.9000,14.8100,34.0522,-118.2437,34.0598,-118.2375,91.0000,America/Los_Angeles,-25200,open_meteo_archive


Đã đọc file: data/m5-forecasting-accuracy/processed/m5_store_dept_daily_with_weather.csv
Shape: 135,870 dòng x 56 cột
Grain kỳ vọng: date + store_id + dept_id
Date range: 2011-01-29 -> 2016-05-22
Coverage: 3 bang, 10 stores, 7 departments


## 2. Kiểm tra schema

Phần này xác nhận các cột bắt buộc có tồn tại không, kiểu dữ liệu có hợp lý không, và các nhóm cột có đúng vai trò phân tích không.

In [2]:
expected_columns = [
    "date", "d", "wm_yr_wk", "store_id", "state_id", "cat_id", "dept_id",
    "daily_revenue", "daily_units", "item_count", "active_item_count",
    "weighted_avg_sell_price", "has_sales", "weekday", "wday", "month",
    "year", "quarter", "week_of_year", "day_of_month", "day_of_year",
    "day_of_week", "day_of_week_num", "is_weekend", "year_month",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "event_count", "snap_CA", "snap_TX", "snap_WI", "snap_active",
    "weather_location_id", "location_name", "weather_spatial_level",
    "weather_code", "temperature_max_c", "temperature_min_c",
    "temperature_mean_c", "apparent_temperature_mean_c", "precipitation_mm",
    "rain_mm", "snowfall_cm", "wind_speed_max_kmh", "wind_gusts_max_kmh",
    "shortwave_radiation_mj_m2", "latitude_requested", "longitude_requested",
    "latitude_open_meteo", "longitude_open_meteo", "elevation_m",
    "timezone", "utc_offset_seconds", "weather_source",
]

missing_expected_cols = sorted(set(expected_columns) - set(df.columns))
extra_cols = sorted(set(df.columns) - set(expected_columns))

column_roles = []
for col in df.columns:
    if col in grain_cols:
        role = "grain/key"
    elif col == target_col:
        role = "target"
    elif col in leakage_risk_cols:
        role = "same-day sales derived / leakage risk"
    elif col in weather_feature_cols:
        role = "weather feature"
    elif col in event_cols or col in ["event_count", "snap_CA", "snap_TX", "snap_WI", "snap_active"]:
        role = "event/SNAP feature"
    elif col in ["weather_location_id", "location_name", "weather_spatial_level", "latitude_requested",
                 "longitude_requested", "latitude_open_meteo", "longitude_open_meteo", "elevation_m",
                 "timezone", "utc_offset_seconds", "weather_source"]:
        role = "weather metadata"
    elif col in ["d", "wm_yr_wk", "weekday", "wday", "month", "year", "quarter", "week_of_year",
                 "day_of_month", "day_of_year", "day_of_week", "day_of_week_num", "is_weekend", "year_month"]:
        role = "calendar/time feature"
    elif col in ["cat_id", "state_id"]:
        role = "hierarchy/location"
    else:
        role = "other"
    column_roles.append({"column": col, "dtype": str(df[col].dtype), "role": role})

schema_summary = pd.DataFrame(
    [
        {"check": "expected_column_count", "value": len(expected_columns)},
        {"check": "actual_column_count", "value": df.shape[1]},
        {"check": "missing_expected_columns", "value": len(missing_expected_cols)},
        {"check": "extra_columns", "value": len(extra_cols)},
        {"check": "date_dtype_is_datetime", "value": pd.api.types.is_datetime64_any_dtype(df["date"])},
        {"check": "target_is_numeric", "value": pd.api.types.is_numeric_dtype(df[target_col])},
    ]
)

dtype_summary = (
    pd.Series(df.dtypes.astype(str), name="dtype")
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="column_count")
)

display(schema_summary)
display(dtype_summary)
display(pd.DataFrame(column_roles))
if missing_expected_cols:
    display(pd.DataFrame({"missing_expected_column": missing_expected_cols}))
if extra_cols:
    display(pd.DataFrame({"extra_column": extra_cols}))

schema_ok = len(missing_expected_cols) == 0 and pd.api.types.is_datetime64_any_dtype(df["date"]) and pd.api.types.is_numeric_dtype(df[target_col])
display(
    Markdown(
        f'''
        **Diễn giải:** Schema {'đạt yêu cầu cơ bản' if schema_ok else 'cần kiểm tra thêm'}.
        File có `{df.shape[1]}` cột; số cột bắt buộc bị thiếu là `{len(missing_expected_cols)}`.
        Cột `date` đã được parse dạng datetime và `daily_revenue` là biến số, phù hợp để kiểm tra chất lượng target.
        Các cột `{', '.join(leakage_risk_cols)}` được đánh dấu là có nguy cơ leakage nếu dùng cho mô hình dự báo trước ngày bán.
        '''
    )
)

,check,value
0,expected_column_count,56
1,actual_column_count,56
2,missing_expected_columns,0
3,extra_columns,0
4,date_dtype_is_datetime,True
5,target_is_numeric,True


,dtype,column_count
0,int64,21
1,str,17
2,float64,17
3,datetime64[us],1


,column,dtype,role
0,date,datetime64[us],grain/key
1,d,str,calendar/time feature
2,wm_yr_wk,int64,calendar/time feature
3,store_id,str,grain/key
4,state_id,str,hierarchy/location
5,cat_id,str,hierarchy/location
6,dept_id,str,grain/key
7,daily_revenue,float64,target
8,daily_units,int64,same-day sales derived / leakage risk
9,item_count,int64,other



        **Diễn giải:** Schema đạt yêu cầu cơ bản.
        File có `56` cột; số cột bắt buộc bị thiếu là `0`.
        Cột `date` đã được parse dạng datetime và `daily_revenue` là biến số, phù hợp để kiểm tra chất lượng target.
        Các cột `daily_units, active_item_count, has_sales, weighted_avg_sell_price` được đánh dấu là có nguy cơ leakage nếu dùng cho mô hình dự báo trước ngày bán.
        

## 3. Kiểm tra missing values

Missing values cần được phân loại theo ngữ cảnh. Missing ở event thường có nghĩa là ngày không có sự kiện, còn missing ở weather hoặc target mới là vấn đề nghiêm trọng hơn.

In [3]:
missing = df.isna().sum().sort_values(ascending=False)
missing_table = missing[missing > 0].to_frame("missing_count")
missing_table["missing_rate"] = missing_table["missing_count"] / len(df)
missing_table = missing_table.reset_index().rename(columns={"index": "column"})

critical_cols = grain_cols + [target_col, "daily_units", "cat_id", "state_id"] + weather_feature_cols
critical_missing = df[critical_cols].isna().sum().sort_values(ascending=False).reset_index()
critical_missing.columns = ["column", "missing_count"]
critical_missing["missing_rate"] = critical_missing["missing_count"] / len(df)

event_missing = df[event_cols].isna().sum().reset_index()
event_missing.columns = ["event_column", "missing_count"]
event_missing["missing_rate"] = event_missing["missing_count"] / len(df)

zero_sales_rows = int((df["daily_units"] == 0).sum())
weighted_avg_missing = int(df["weighted_avg_sell_price"].isna().sum())
weighted_avg_missing_when_zero_units = int(df.loc[df["daily_units"] == 0, "weighted_avg_sell_price"].isna().sum())
weighted_avg_missing_when_positive_units = int(df.loc[df["daily_units"] > 0, "weighted_avg_sell_price"].isna().sum())

missing_interpretation = pd.DataFrame(
    [
        {
            "check": "critical_missing_total",
            "value": int(df[critical_cols].isna().sum().sum()),
            "interpretation": "Nếu > 0 thì có vấn đề ở key/target/weather chính.",
        },
        {
            "check": "event_missing_total",
            "value": int(df[event_cols].isna().sum().sum()),
            "interpretation": "Missing event thường biểu thị ngày không có event, không tự động là lỗi.",
        },
        {
            "check": "weighted_avg_sell_price_missing",
            "value": weighted_avg_missing,
            "interpretation": "Có thể chấp nhận nếu chỉ xảy ra ở ngày/nhóm không bán units.",
        },
        {
            "check": "weighted_avg_missing_when_zero_units",
            "value": weighted_avg_missing_when_zero_units,
            "interpretation": "Missing hợp lý vì không thể tính average price nếu daily_units = 0.",
        },
        {
            "check": "weighted_avg_missing_when_positive_units",
            "value": weighted_avg_missing_when_positive_units,
            "interpretation": "Nếu > 0 thì có lỗi tính revenue/price.",
        },
    ]
)

display(missing_table)
display(critical_missing)
display(event_missing)
display(missing_interpretation)

serious_missing = int(df[critical_cols].isna().sum().sum()) + weighted_avg_missing_when_positive_units
display(
    Markdown(
        f'''
        **Diễn giải:** Missing value nghiêm trọng ở key, target và weather chính là `{int(df[critical_cols].isna().sum().sum()):,}`.
        `weighted_avg_sell_price` thiếu `{weighted_avg_missing:,}` dòng, nhưng thiếu ở dòng có `daily_units > 0` là
        `{weighted_avg_missing_when_positive_units:,}`. Vì vậy missing hiện tại chủ yếu đến từ cột event và trường hợp không có sales,
        không phải lỗi chặn phân tích.
        '''
    )
)

,column,missing_count,missing_rate
0,event_type_2,135590,0.9979
1,event_name_2,135590,0.9979
2,event_type_1,124810,0.9186
3,event_name_1,124810,0.9186
4,weighted_avg_sell_price,363,0.0027


,column,missing_count,missing_rate
0,date,0,0.0000
1,store_id,0,0.0000
2,wind_gusts_max_kmh,0,0.0000
3,wind_speed_max_kmh,0,0.0000
4,snowfall_cm,0,0.0000
5,rain_mm,0,0.0000
6,precipitation_mm,0,0.0000
7,apparent_temperature_mean_c,0,0.0000
8,temperature_mean_c,0,0.0000
9,temperature_min_c,0,0.0000


,event_column,missing_count,missing_rate
0,event_name_1,124810,0.9186
1,event_type_1,124810,0.9186
2,event_name_2,135590,0.9979
3,event_type_2,135590,0.9979


,check,value,interpretation
0,critical_missing_total,0,Nếu > 0 thì có vấn đề ở key/target/weather chính.
1,event_missing_total,520800,Missing event thường biểu thị ngày không có ev...
2,weighted_avg_sell_price_missing,363,Có thể chấp nhận nếu chỉ xảy ra ở ngày/nhóm kh...
3,weighted_avg_missing_when_zero_units,363,Missing hợp lý vì không thể tính average price...
4,weighted_avg_missing_when_positive_units,0,Nếu > 0 thì có lỗi tính revenue/price.



        **Diễn giải:** Missing value nghiêm trọng ở key, target và weather chính là `0`.
        `weighted_avg_sell_price` thiếu `363` dòng, nhưng thiếu ở dòng có `daily_units > 0` là
        `0`. Vì vậy missing hiện tại chủ yếu đến từ cột event và trường hợp không có sales,
        không phải lỗi chặn phân tích.
        

## 4. Kiểm tra duplicate và key uniqueness

Grain chính của bảng là `date + store_id + dept_id`. Nếu key này bị duplicate thì target ở cùng một đơn vị phân tích bị lặp, làm sai EDA và modeling.

In [4]:
full_duplicate_rows = int(df.duplicated().sum())
grain_duplicate_rows = int(df.duplicated(grain_cols).sum())
date_state_weather_meta_dups = int(
    df[["date", "state_id", "weather_location_id", "location_name"]]
    .drop_duplicates()
    .duplicated(["date", "state_id"])
    .sum()
)

date_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_dates = sorted(set(date_range) - set(df["date"].drop_duplicates()))

rows_per_date_store = (
    df.groupby(["date", "store_id"], observed=True)
    .size()
    .rename("rows_per_date_store")
    .reset_index()
)
rows_per_date_store_summary = rows_per_date_store["rows_per_date_store"].describe().to_frame("rows_per_date_store")

expected_rows = df["date"].nunique() * df["store_id"].nunique() * df["dept_id"].nunique()
key_summary = pd.DataFrame(
    [
        {"check": "full_duplicate_rows", "value": full_duplicate_rows},
        {"check": "duplicate_date_store_dept_keys", "value": grain_duplicate_rows},
        {"check": "weather_metadata_duplicate_date_state_keys", "value": date_state_weather_meta_dups},
        {"check": "unique_dates", "value": df["date"].nunique()},
        {"check": "continuous_expected_dates", "value": len(date_range)},
        {"check": "missing_dates_in_continuous_range", "value": len(missing_dates)},
        {"check": "expected_rows_dates_x_stores_x_depts", "value": expected_rows},
        {"check": "actual_rows", "value": len(df)},
        {"check": "actual_minus_expected_rows", "value": len(df) - expected_rows},
    ]
)

display(key_summary)
display(rows_per_date_store_summary)
display(rows_per_date_store["rows_per_date_store"].value_counts().rename_axis("rows_per_date_store").reset_index(name="date_store_count"))

display(
    Markdown(
        f'''
        **Diễn giải:** Key `date + store_id + dept_id` có `{grain_duplicate_rows}` dòng duplicate.
        Mỗi cặp `date + store_id` có `{int(rows_per_date_store['rows_per_date_store'].min())}` đến
        `{int(rows_per_date_store['rows_per_date_store'].max())}` rows, đúng kỳ vọng là một row cho mỗi department.
        Số ngày thiếu trong khoảng liên tục là `{len(missing_dates)}`. Như vậy bảng giữ được grain ổn định để phân tích.
        '''
    )
)

,check,value
0,full_duplicate_rows,0
1,duplicate_date_store_dept_keys,0
2,weather_metadata_duplicate_date_state_keys,0
3,unique_dates,1941
4,continuous_expected_dates,1941
5,missing_dates_in_continuous_range,0
6,expected_rows_dates_x_stores_x_depts,135870
7,actual_rows,135870
8,actual_minus_expected_rows,0


,rows_per_date_store
count,"19,410.0000"
mean,7.0000
std,0.0000
min,7.0000
25%,7.0000
50%,7.0000
75%,7.0000
max,7.0000


,rows_per_date_store,date_store_count
0,7,19410



        **Diễn giải:** Key `date + store_id + dept_id` có `0` dòng duplicate.
        Mỗi cặp `date + store_id` có `7` đến
        `7` rows, đúng kỳ vọng là một row cho mỗi department.
        Số ngày thiếu trong khoảng liên tục là `0`. Như vậy bảng giữ được grain ổn định để phân tích.
        

## 5. Kiểm tra range và giá trị bất thường

Phần này kiểm tra các điều kiện hợp lệ cơ bản: revenue không âm, units không âm, active item không vượt item_count, cờ binary đúng 0/1, weather nằm trong khoảng hợp lý, và các outlier revenue cần được gắn cờ thay vì xóa vội.

In [5]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_describe = df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

binary_cols = ["has_sales", "is_weekend", "snap_CA", "snap_TX", "snap_WI", "snap_active"]
binary_value_summary = []
for col in binary_cols:
    values = sorted(df[col].dropna().unique().tolist())
    invalid_count = int((~df[col].isin([0, 1])).sum())
    binary_value_summary.append({"column": col, "unique_values": values, "invalid_binary_count": invalid_count})

range_checks = pd.DataFrame(
    [
        {"check": "negative_daily_revenue_rows", "value": int((df["daily_revenue"] < 0).sum())},
        {"check": "zero_daily_revenue_rows", "value": int((df["daily_revenue"] == 0).sum())},
        {"check": "negative_daily_units_rows", "value": int((df["daily_units"] < 0).sum())},
        {"check": "active_item_gt_item_count_rows", "value": int((df["active_item_count"] > df["item_count"]).sum())},
        {"check": "has_sales_inconsistent_with_units_rows", "value": int((df["has_sales"] != (df["daily_units"] > 0).astype(int)).sum())},
        {"check": "weighted_price_nonpositive_when_sales_rows", "value": int(((df["daily_units"] > 0) & (df["weighted_avg_sell_price"] <= 0)).sum())},
        {"check": "temp_min_gt_mean_rows", "value": int((df["temperature_min_c"] > df["temperature_mean_c"]).sum())},
        {"check": "temp_mean_gt_max_rows", "value": int((df["temperature_mean_c"] > df["temperature_max_c"]).sum())},
        {"check": "negative_precipitation_rows", "value": int((df["precipitation_mm"] < 0).sum())},
        {"check": "negative_rain_rows", "value": int((df["rain_mm"] < 0).sum())},
        {"check": "negative_snowfall_rows", "value": int((df["snowfall_cm"] < 0).sum())},
        {"check": "negative_wind_speed_rows", "value": int((df["wind_speed_max_kmh"] < 0).sum())},
        {"check": "negative_radiation_rows", "value": int((df["shortwave_radiation_mj_m2"] < 0).sum())},
        {"check": "weather_code_outside_0_99_rows", "value": int(((df["weather_code"] < 0) | (df["weather_code"] > 99)).sum())},
    ]
)

def iqr_outlier_summary(data, col):
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((data[col] < lower) | (data[col] > upper)).sum())
    return q1, q3, lower, upper, count

q1, q3, lower, upper, outlier_count = iqr_outlier_summary(df, "daily_revenue")
revenue_outlier_summary = pd.DataFrame(
    [
        {"metric": "q1", "value": q1},
        {"metric": "q3", "value": q3},
        {"metric": "iqr_lower_bound", "value": lower},
        {"metric": "iqr_upper_bound", "value": upper},
        {"metric": "iqr_outlier_rows", "value": outlier_count},
        {"metric": "iqr_outlier_rate", "value": outlier_count / len(df)},
    ]
)

top_revenue_rows = (
    df.sort_values("daily_revenue", ascending=False)
    [["date", "store_id", "state_id", "cat_id", "dept_id", "daily_revenue", "daily_units",
      "weighted_avg_sell_price", "event_name_1", "temperature_mean_c", "precipitation_mm"]]
    .head(15)
)

display(range_checks)
display(pd.DataFrame(binary_value_summary))
display(numeric_describe.loc[["daily_revenue", "daily_units", "item_count", "active_item_count", "weighted_avg_sell_price"] + weather_feature_cols])
display(revenue_outlier_summary)
display(top_revenue_rows)

serious_range_issues = int(range_checks.loc[~range_checks["check"].isin(["zero_daily_revenue_rows"]), "value"].sum())
display(
    Markdown(
        f'''
        **Diễn giải:** Không nên xem tất cả outlier revenue là lỗi, vì bán lẻ thường có đỉnh doanh thu thật theo store/department/date.
        Các kiểm tra range nghiêm trọng ngoài zero revenue có tổng `{serious_range_issues}` dòng vi phạm.
        Có `{int((df['daily_revenue'] == 0).sum()):,}` dòng zero revenue; đây là trạng thái hợp lệ nếu department-store không bán trong ngày.
        Theo IQR toàn cục, có `{outlier_count:,}` dòng doanh thu nằm ngoài ngưỡng cao/thấp; các dòng này nên được phân tích trong EDA thay vì xóa tự động.
        '''
    )
)

,check,value
0,negative_daily_revenue_rows,0
1,zero_daily_revenue_rows,363
2,negative_daily_units_rows,0
3,active_item_gt_item_count_rows,0
4,has_sales_inconsistent_with_units_rows,0
5,weighted_price_nonpositive_when_sales_rows,0
6,temp_min_gt_mean_rows,0
7,temp_mean_gt_max_rows,0
8,negative_precipitation_rows,0
9,negative_rain_rows,0


,column,unique_values,invalid_binary_count
0,has_sales,"[0, 1]",0
1,is_weekend,"[0, 1]",0
2,snap_CA,"[0, 1]",0
3,snap_TX,"[0, 1]",0
4,snap_WI,"[0, 1]",0
5,snap_active,"[0, 1]",0


,count,mean,std,min,1%,5%,50%,95%,99%,max
daily_revenue,"135,870.0000","1,410.0062","1,384.2343",0.0000,15.3800,42.6345,985.7900,"4,258.8290","6,484.8792","11,198.9500"
daily_units,"135,870.0000",492.5824,603.8029,0.0000,6.0000,18.0000,288.0000,"1,841.5500","2,878.3100","5,118.0000"
item_count,"135,870.0000",435.5714,206.2730,149.0000,149.0000,149.0000,416.0000,823.0000,823.0000,823.0000
active_item_count,"135,870.0000",139.3926,107.7646,0.0000,5.0000,12.0000,111.0000,370.0000,471.0000,621.0000
weighted_avg_sell_price,"135,507.0000",3.2682,1.0581,0.2000,1.4988,1.9266,3.1447,5.1735,5.7246,10.9650
weather_code,"135,870.0000",19.6222,26.3339,0.0000,0.0000,0.0000,3.0000,65.0000,73.0000,75.0000
temperature_max_c,"135,870.0000",20.8922,10.7769,-17.6000,-7.7000,0.3000,22.7000,35.7000,38.8000,42.4000
temperature_min_c,"135,870.0000",11.1111,9.0944,-25.3000,-15.8000,-5.3000,11.6000,24.9000,28.1000,31.7000
temperature_mean_c,"135,870.0000",15.7061,9.6775,-22.9000,-11.2000,-2.5000,16.8000,29.8000,33.3000,37.2000
apparent_temperature_mean_c,"135,870.0000",14.1394,11.8188,-31.4000,-17.7000,-7.8000,15.2000,31.8000,35.3000,38.6000


,metric,value
0,q1,500.7100
1,q3,"1,900.5075"
2,iqr_lower_bound,"-1,598.9862"
3,iqr_upper_bound,"4,000.2037"
4,iqr_outlier_rows,"8,194.0000"
5,iqr_outlier_rate,0.0603


,date,store_id,state_id,cat_id,dept_id,daily_revenue,daily_units,weighted_avg_sell_price,event_name_1,temperature_mean_c,precipitation_mm
79466,2014-03-09,CA_3,CA,FOODS,FOODS_3,"11,198.9500",4864,2.3024,NaN,18.5000,0.0000
81426,2014-04-06,CA_3,CA,FOODS,FOODS_3,"10,622.9700",4721,2.2502,NaN,18.8000,0.0000
82336,2014-04-19,CA_3,CA,FOODS,FOODS_3,"10,421.9600",4527,2.3022,NaN,18.6000,0.0000
87586,2014-07-03,CA_3,CA,FOODS,FOODS_3,"10,376.0000",4826,2.1500,NaN,25.7000,0.0000
94096,2014-10-04,CA_3,CA,FOODS,FOODS_3,"10,296.4700",4296,2.3968,EidAlAdha,28.1000,0.0000
94166,2014-10-05,CA_3,CA,FOODS,FOODS_3,"10,295.3800",4456,2.3105,NaN,26.1000,0.0000
82406,2014-04-20,CA_3,CA,FOODS,FOODS_3,"10,227.1400",4365,2.3430,Easter,19.5000,0.0000
82896,2014-04-27,CA_3,CA,FOODS,FOODS_3,"10,216.7200",4438,2.3021,NaN,16.1000,0.0000
104946,2015-03-08,CA_3,CA,FOODS,FOODS_3,"10,215.8300",4211,2.4260,NaN,17.6000,0.0000
81916,2014-04-13,CA_3,CA,FOODS,FOODS_3,"10,188.2100",4355,2.3394,NaN,14.5000,0.0000



        **Diễn giải:** Không nên xem tất cả outlier revenue là lỗi, vì bán lẻ thường có đỉnh doanh thu thật theo store/department/date.
        Các kiểm tra range nghiêm trọng ngoài zero revenue có tổng `0` dòng vi phạm.
        Có `363` dòng zero revenue; đây là trạng thái hợp lệ nếu department-store không bán trong ngày.
        Theo IQR toàn cục, có `8,194` dòng doanh thu nằm ngoài ngưỡng cao/thấp; các dòng này nên được phân tích trong EDA thay vì xóa tự động.
        

## 6. Data quality assessment

Tổng hợp các phát hiện thành bảng đánh giá chất lượng. Mục tiêu là quyết định dữ liệu có đủ tốt để chuyển sang EDA/model-ready hay cần xử lý lỗi trước.

In [6]:
quality_items = [
    {
        "dimension": "Completeness",
        "status": "OK" if int(df[critical_cols].isna().sum().sum()) == 0 else "WARN",
        "evidence": f"Missing ở key/target/weather chính: {int(df[critical_cols].isna().sum().sum()):,}. Event missing được xem là không có event.",
        "action": "Không cần impute các cột event trước EDA; khi model có thể tạo has_event/event_count.",
    },
    {
        "dimension": "Uniqueness",
        "status": "OK" if grain_duplicate_rows == 0 and full_duplicate_rows == 0 else "WARN",
        "evidence": f"Duplicate full rows: {full_duplicate_rows:,}; duplicate grain date+store+dept: {grain_duplicate_rows:,}.",
        "action": "Có thể dùng grain date+store_id+dept_id làm đơn vị phân tích chính.",
    },
    {
        "dimension": "Validity",
        "status": "OK" if serious_range_issues == 0 else "WARN",
        "evidence": f"Range violations nghiêm trọng ngoài zero revenue: {serious_range_issues:,}.",
        "action": "Giữ outlier revenue để EDA, không xóa tự động.",
    },
    {
        "dimension": "Consistency",
        "status": "OK" if int((df["has_sales"] != (df["daily_units"] > 0).astype(int)).sum()) == 0 else "WARN",
        "evidence": "has_sales nhất quán với daily_units; active_item_count không vượt item_count.",
        "action": "Các cột same-day sales chỉ dùng EDA, không dùng trực tiếp cho model dự báo.",
    },
    {
        "dimension": "Temporal coverage",
        "status": "OK" if len(missing_dates) == 0 else "WARN",
        "evidence": f"Khoảng ngày liên tục: {df['date'].min().date()} đến {df['date'].max().date()}, thiếu {len(missing_dates)} ngày.",
        "action": "Có thể chia train/test theo thời gian, không shuffle.",
    },
    {
        "dimension": "Weather join quality",
        "status": "OK" if int(df[weather_feature_cols].isna().sum().sum()) == 0 else "WARN",
        "evidence": f"Missing weather feature: {int(df[weather_feature_cols].isna().sum().sum()):,}. Weather ở mức date+state_id.",
        "action": "Ghi rõ limitation: weather dùng thành phố đại diện cho state, không phải tọa độ store thật.",
    },
    {
        "dimension": "Target construction",
        "status": "WARN",
        "evidence": "daily_revenue là revenue tái tạo từ daily_units * weekly sell_price, không phải invoice revenue trực tiếp.",
        "action": "Trong report/modeling phải ghi rõ measurement limitation này.",
    },
    {
        "dimension": "Leakage risk",
        "status": "WARN",
        "evidence": f"Các cột có nguy cơ leakage: {', '.join(leakage_risk_cols)}.",
        "action": "Loại khỏi model-ready hoặc chỉ dùng dạng lag/rolling từ quá khứ.",
    },
]
quality_scorecard = pd.DataFrame(quality_items)
display(quality_scorecard)

ok_count = int((quality_scorecard["status"] == "OK").sum())
warn_count = int((quality_scorecard["status"] == "WARN").sum())
display(
    Markdown(
        f'''
        **Kết luận data quality:** Có `{ok_count}` nhóm kiểm tra đạt `OK` và `{warn_count}` nhóm cần ghi chú `WARN`.
        Các cảnh báo hiện tại chủ yếu là limitation/phòng leakage, không phải lỗi kỹ thuật chặn EDA.
        Bảng đã đủ điều kiện để chuyển sang EDA, với điều kiện giữ rõ các cột nào không an toàn cho mô hình dự báo.
        '''
    )
)

,dimension,status,evidence,action
0,Completeness,OK,Missing ở key/target/weather chính: 0. Event m...,Không cần impute các cột event trước EDA; khi ...
1,Uniqueness,OK,Duplicate full rows: 0; duplicate grain date+s...,Có thể dùng grain date+store_id+dept_id làm đơ...
2,Validity,OK,Range violations nghiêm trọng ngoài zero reven...,"Giữ outlier revenue để EDA, không xóa tự động."
3,Consistency,OK,has_sales nhất quán với daily_units; active_it...,"Các cột same-day sales chỉ dùng EDA, không dùn..."
4,Temporal coverage,OK,Khoảng ngày liên tục: 2011-01-29 đến 2016-05-2...,"Có thể chia train/test theo thời gian, không s..."
5,Weather join quality,OK,Missing weather feature: 0. Weather ở mức date...,Ghi rõ limitation: weather dùng thành phố đại ...
6,Target construction,WARN,daily_revenue là revenue tái tạo từ daily_unit...,Trong report/modeling phải ghi rõ measurement ...
7,Leakage risk,WARN,"Các cột có nguy cơ leakage: daily_units, activ...",Loại khỏi model-ready hoặc chỉ dùng dạng lag/r...



        **Kết luận data quality:** Có `6` nhóm kiểm tra đạt `OK` và `2` nhóm cần ghi chú `WARN`.
        Các cảnh báo hiện tại chủ yếu là limitation/phòng leakage, không phải lỗi kỹ thuật chặn EDA.
        Bảng đã đủ điều kiện để chuyển sang EDA, với điều kiện giữ rõ các cột nào không an toàn cho mô hình dự báo.
        

## 7. Bias, limitation và variation assessment

Trước khi EDA, cần ghi rõ dữ liệu nói được điều gì và không nói được điều gì. Phần này không kết luận causal effect; chỉ đánh giá phạm vi, sai lệch đo lường và độ biến động quan trọng.

In [7]:
bias_limitation_table = pd.DataFrame(
    [
        {
            "type": "Coverage bias",
            "risk": "M5 chỉ gồm 3 bang CA, TX, WI và 10 stores.",
            "impact": "Không nên suy rộng kết quả cho toàn bộ thị trường retail Mỹ.",
            "mitigation": "Giới hạn kết luận trong phạm vi M5; khi so sánh Maven cần nói rõ dataset shift.",
        },
        {
            "type": "Selection bias",
            "risk": "Không biết đầy đủ quy trình chọn store/item trong M5.",
            "impact": "Store/item trong dữ liệu có thể không đại diện cho tất cả cửa hàng/sản phẩm.",
            "mitigation": "Không diễn giải như mẫu ngẫu nhiên của toàn bộ Walmart/retail.",
        },
        {
            "type": "Measurement limitation",
            "risk": "daily_revenue được tái tạo từ units và weekly price.",
            "impact": "Không phản ánh transaction-level discounts, coupons, hoặc giá thay đổi trong ngày nếu có.",
            "mitigation": "Gọi là reconstructed revenue hoặc estimated revenue trong report.",
        },
        {
            "type": "Weather spatial limitation",
            "risk": "Weather join ở date + state_id, dùng representative city cho mỗi bang.",
            "impact": "Weather có thể lệch với thời tiết thực tại từng store.",
            "mitigation": "Xem weather là external context, không phải đo lường chính xác store-level.",
        },
        {
            "type": "Temporal limitation",
            "risk": "Dữ liệu giai đoạn 2011-2016.",
            "impact": "Hành vi retail hiện tại có thể khác do inflation, ecommerce, COVID/post-COVID.",
            "mitigation": "Không overclaim cho thị trường hiện tại.",
        },
        {
            "type": "Aggregation limitation",
            "risk": "Grain chính là department, không phải từng item.",
            "impact": "Mất dị biệt giữa các SKU trong cùng department.",
            "mitigation": "Phù hợp cho model chính; item-level có thể là hướng mở rộng.",
        },
        {
            "type": "Forecasting leakage risk",
            "risk": "daily_units, has_sales, active_item_count, weighted_avg_sell_price biết sau khi ngày bán xảy ra.",
            "impact": "Nếu dùng trực tiếp, model dự báo sẽ bị leakage.",
            "mitigation": "Chỉ dùng EDA hoặc tạo lag/rolling dựa trên quá khứ.",
        },
    ]
)

revenue_by_state = (
    df.groupby("state_id", observed=True)["daily_revenue"]
    .agg(["count", "mean", "std", "min", "median", "max", "sum"])
    .reset_index()
)
revenue_by_state["cv"] = revenue_by_state["std"] / revenue_by_state["mean"]

revenue_by_dept = (
    df.groupby(["cat_id", "dept_id"], observed=True)["daily_revenue"]
    .agg(["count", "mean", "std", "min", "median", "max", "sum"])
    .reset_index()
    .sort_values("sum", ascending=False)
)
revenue_by_dept["cv"] = revenue_by_dept["std"] / revenue_by_dept["mean"]

revenue_by_store = (
    df.groupby(["state_id", "store_id"], observed=True)["daily_revenue"]
    .agg(["count", "mean", "std", "min", "median", "max", "sum"])
    .reset_index()
    .sort_values("sum", ascending=False)
)
revenue_by_store["cv"] = revenue_by_store["std"] / revenue_by_store["mean"]

zero_rate_by_dept = (
    df.assign(is_zero_revenue=(df["daily_revenue"] == 0).astype(int))
    .groupby(["cat_id", "dept_id"], observed=True)["is_zero_revenue"]
    .agg(["count", "sum", "mean"])
    .reset_index()
    .rename(columns={"sum": "zero_revenue_rows", "mean": "zero_revenue_rate"})
    .sort_values("zero_revenue_rate", ascending=False)
)

weather_variation_by_state = (
    df[["date", "state_id"] + weather_feature_cols]
    .drop_duplicates(["date", "state_id"])
    .groupby("state_id", observed=True)
    .agg(
        temp_mean_avg=("temperature_mean_c", "mean"),
        temp_mean_std=("temperature_mean_c", "std"),
        temp_mean_min=("temperature_mean_c", "min"),
        temp_mean_max=("temperature_mean_c", "max"),
        precipitation_avg=("precipitation_mm", "mean"),
        precipitation_std=("precipitation_mm", "std"),
        precipitation_max=("precipitation_mm", "max"),
        wind_speed_avg=("wind_speed_max_kmh", "mean"),
        wind_speed_max=("wind_speed_max_kmh", "max"),
    )
    .reset_index()
)

yearly_coverage = (
    df.groupby("year", observed=True)
    .agg(
        rows=("daily_revenue", "size"),
        date_count=("date", "nunique"),
        revenue_sum=("daily_revenue", "sum"),
        revenue_mean=("daily_revenue", "mean"),
    )
    .reset_index()
)

display(bias_limitation_table)
display(revenue_by_state)
display(revenue_by_store)
display(revenue_by_dept)
display(zero_rate_by_dept)
display(weather_variation_by_state)
display(yearly_coverage)

highest_cv_dept = revenue_by_dept.sort_values("cv", ascending=False).iloc[0]
top_revenue_dept = revenue_by_dept.sort_values("sum", ascending=False).iloc[0]
display(
    Markdown(
        f'''
        **Diễn giải variation:** Revenue biến động rõ theo state, store và department, nên EDA/modeling cần giữ các chiều này.
        Department đóng góp revenue lớn nhất là `{top_revenue_dept['dept_id']}` với tổng revenue khoảng `{top_revenue_dept['sum']:,.0f}`.
        Department có hệ số biến thiên revenue cao nhất là `{highest_cv_dept['dept_id']}` với CV khoảng `{highest_cv_dept['cv']:.2f}`.
        Weather cũng khác nhau theo state, nhưng vì weather đang ở mức representative city, phần này nên được diễn giải thận trọng.
        '''
    )
)

,type,risk,impact,mitigation
0,Coverage bias,"M5 chỉ gồm 3 bang CA, TX, WI và 10 stores.",Không nên suy rộng kết quả cho toàn bộ thị trư...,Giới hạn kết luận trong phạm vi M5; khi so sán...
1,Selection bias,Không biết đầy đủ quy trình chọn store/item tr...,Store/item trong dữ liệu có thể không đại diện...,Không diễn giải như mẫu ngẫu nhiên của toàn bộ...
2,Measurement limitation,daily_revenue được tái tạo từ units và weekly ...,"Không phản ánh transaction-level discounts, co...",Gọi là reconstructed revenue hoặc estimated re...
3,Weather spatial limitation,"Weather join ở date + state_id, dùng represent...",Weather có thể lệch với thời tiết thực tại từn...,"Xem weather là external context, không phải đo..."
4,Temporal limitation,Dữ liệu giai đoạn 2011-2016.,Hành vi retail hiện tại có thể khác do inflati...,Không overclaim cho thị trường hiện tại.
5,Aggregation limitation,"Grain chính là department, không phải từng item.",Mất dị biệt giữa các SKU trong cùng department.,Phù hợp cho model chính; item-level có thể là ...
6,Forecasting leakage risk,"daily_units, has_sales, active_item_count, wei...","Nếu dùng trực tiếp, model dự báo sẽ bị leakage.",Chỉ dùng EDA hoặc tạo lag/rolling dựa trên quá...


,state_id,count,mean,std,min,median,max,sum,cv
0,CA,54348,"1,581.7943","1,582.7561",0.0000,"1,142.6450","11,198.9500","85,967,359.0506",1.0006
1,TX,40761,"1,352.2888","1,218.3125",0.0000,934.3900,"9,947.5500","55,120,642.0371",0.9009
2,WI,40761,"1,238.6729","1,220.9124",0.0000,807.5600,"8,944.6000","50,489,545.0269",0.9857


,state_id,store_id,count,mean,std,min,median,max,sum,cv
2,CA,CA_3,13587,"2,406.6488","2,145.6796",0.0000,"1,692.8200","11,198.9500","32,699,137.6831",0.8916
0,CA,CA_1,13587,"1,689.4299","1,572.6592",0.0000,"1,231.9300","9,015.7300","22,954,283.7756",0.9309
5,TX,TX_2,13587,"1,537.7200","1,369.5680",0.0000,"1,092.2900","9,947.5500","20,893,002.1948",0.8906
6,TX,TX_3,13587,"1,338.8017","1,167.6382",0.0000,938.1500,"9,610.0400","18,190,298.6966",0.8722
8,WI,WI_2,13587,"1,334.5098","1,367.4082",0.0000,765.8500,"8,944.6000","18,131,985.0430",1.0247
1,CA,CA_2,13587,"1,313.6240","1,070.5773",0.0000,"1,089.1200","7,691.8300","17,848,209.3906",0.8150
9,WI,WI_3,13587,"1,269.6052","1,261.1953",0.0000,779.9600,"7,865.7803","17,250,126.1765",0.9934
4,TX,TX_1,13587,"1,180.3445","1,072.3072",0.0000,795.7500,"6,740.3000","16,037,341.1457",0.9085
7,WI,WI_1,13587,"1,111.9036",992.7073,0.0000,877.0700,"7,350.7300","15,107,433.8073",0.8928
3,CA,CA_4,13587,917.4747,769.6703,0.0000,711.1000,"4,067.5800","12,465,728.2013",0.8389


,cat_id,dept_id,count,mean,std,min,median,max,sum,cv
2,FOODS,FOODS_3,19410,"3,727.2470","1,553.0280",0.0000,"3,440.6800","11,198.9500","72,345,863.7778",0.4167
5,HOUSEHOLD,HOUSEHOLD_1,19410,"2,170.6738","1,067.0314",0.0000,"1,963.5200","8,464.7600","42,132,778.5504",0.4916
1,FOODS,FOODS_2,19410,"1,318.5145",737.9398,0.0000,"1,226.6250","7,082.8200","25,592,365.6414",0.5597
3,HOBBIES,HOBBIES_1,19410,"1,139.6314",532.6740,0.0000,"1,025.9850","3,804.0300","22,120,244.5109",0.4674
6,HOUSEHOLD,HOUSEHOLD_2,19410,771.9267,403.5542,0.0000,653.3250,"3,115.3900","14,983,097.8670",0.5228
0,FOODS,FOODS_1,19410,680.1544,288.1641,0.0000,631.0450,"2,618.3300","13,201,796.0601",0.4237
4,HOBBIES,HOBBIES_2,19410,61.8959,36.1287,0.0000,55.7650,306.6900,"1,201,399.7070",0.5837


,cat_id,dept_id,count,zero_revenue_rows,zero_revenue_rate
4,HOBBIES,HOBBIES_2,19410,81,0.0042
1,FOODS,FOODS_2,19410,56,0.0029
6,HOUSEHOLD,HOUSEHOLD_2,19410,52,0.0027
3,HOBBIES,HOBBIES_1,19410,51,0.0026
5,HOUSEHOLD,HOUSEHOLD_1,19410,51,0.0026
0,FOODS,FOODS_1,19410,48,0.0025
2,FOODS,FOODS_3,19410,24,0.0012


,state_id,temp_mean_avg,temp_mean_std,temp_mean_min,temp_mean_max,precipitation_avg,precipitation_std,precipitation_max,wind_speed_avg,wind_speed_max
0,CA,18.3414,5.4022,4.8000,31.8000,0.5482,3.0450,54.1000,12.7814,33.0000
1,TX,19.5505,9.0546,-7.3000,37.2000,2.5102,8.0365,85.5000,20.3795,45.3000
2,WI,8.3481,10.6406,-22.9000,31.9000,2.4117,5.5950,53.7000,22.3320,48.5000


,year,rows,date_count,revenue_sum,revenue_mean
0,2011,23590,337,"23,891,336.0950","1,012.7739"
1,2012,25620,366,"32,649,200.8746","1,274.3638"
2,2013,25550,365,"35,923,373.3411","1,406.0029"
3,2014,25550,365,"37,861,913.2164","1,481.8753"
4,2015,25550,365,"42,416,456.5945","1,660.1353"
5,2016,10010,143,"18,835,265.9931","1,881.6450"



        **Diễn giải variation:** Revenue biến động rõ theo state, store và department, nên EDA/modeling cần giữ các chiều này.
        Department đóng góp revenue lớn nhất là `FOODS_3` với tổng revenue khoảng `72,345,864`.
        Department có hệ số biến thiên revenue cao nhất là `HOBBIES_2` với CV khoảng `0.58`.
        Weather cũng khác nhau theo state, nhưng vì weather đang ở mức representative city, phần này nên được diễn giải thận trọng.
        

## 8. Kết luận trước EDA

Notebook này cho thấy dữ liệu đã đủ điều kiện để bước sang EDA, nhưng có một số giới hạn phải giữ trong report/modeling:

- Grain `date + store_id + dept_id` ổn định và không duplicate.
- Missing nghiêm trọng ở key, target và weather chính không xuất hiện.
- Missing event chủ yếu là ngày không có sự kiện, không nên xem là lỗi.
- `daily_revenue` không âm và có zero revenue hợp lệ.
- Outlier revenue cần được phân tích trong EDA, không xóa tự động.
- Weather được join ở mức bang/state bằng representative city, nên không phải weather chính xác tại từng store.
- `daily_units`, `active_item_count`, `has_sales`, `weighted_avg_sell_price` là cột có nguy cơ leakage nếu dùng trực tiếp trong mô hình dự báo.

Bước tiếp theo nên là EDA có mục tiêu:

```text
daily_revenue theo thời gian, store, state, category, department,
event/SNAP, weather, zero revenue và outlier revenue.
```

Sau EDA mới tạo file model-ready với lag/rolling features và loại các cột leakage.